# Logistic Regression para estudiantes con Pipeline + MLflow

Este notebook entrena un modelo de logistic regression usando el dataset `estudiantes.csv`.

La diferencia importante frente al notebook anterior es que ahora se guarda en MLflow un `Pipeline` completo:

```text
datos originales → preprocesamiento → logistic regression
```

De esta manera, Flask o Streamlit pueden enviar las columnas originales del dataset, sin tener que enviar columnas codificadas manualmente.


## 1. Importación de librerías

In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from mlflow.models.signature import infer_signature

from sklearn.linear_model import LogisticRegression

import mlflow
import mlflow.sklearn


## 2. Configuración de MLflow



In [15]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")
mlflow.set_experiment("linear_regretion_pipeline")


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778910946266, experiment_id='1', last_update_time=1778910946266, lifecycle_stage='active', name='linear_regretion_pipeline', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [16]:
df = pd.read_csv("data/estudiantes.csv", delimiter=",")
df.head()


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


In [17]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (10000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      10000 non-null  object 
 1   modalidad    10000 non-null  object 
 2   beca         10000 non-null  object 
 3   edad         10000 non-null  int64  
 4   promedio     10000 non-null  float64
 5   asistencias  10000 non-null  int64  
 6   aprobado     10000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 547.0+ KB


## 4. Preparación de datos

La variable objetivo `aprobado` se transforma:

- `Si` → `1`
- `No` → `0`

Además, se elimina `beca` de las variables predictoras porque no afecta.


In [18]:
df["aprobado"] = (
    df["aprobado"]
    .str.strip()
    .str.lower()
    .map({
        "si": 1,
        "no": 0
    })
)

X = df.drop(columns=["aprobado","beca"])
y = df["aprobado"]

print("Columnas originales usadas para entrenamiento:")
print(X.columns.tolist())

print("\nDistribución de la variable objetivo:")
print(y.value_counts())


Columnas originales usadas para entrenamiento:
['carrera', 'modalidad', 'edad', 'promedio', 'asistencias']

Distribución de la variable objetivo:
aprobado
0    5844
1    4156
Name: count, dtype: int64


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [19]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['carrera', 'modalidad']

Columnas numéricas:
['edad', 'promedio', 'asistencias']


## 6. División de datos

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (8000, 5)
Prueba: (2000, 5)


## 7. Creación del Pipeline

Este es el punto clave para producción.

El `Pipeline` contiene:

1. `preprocessor`: transforma variables categóricas con `OneHotEncoder`.
2. `modelo`: entrena el árbol de decisión.

Cuando guardamos este `Pipeline` en MLflow, Streamlit podrá enviar datos originales como `job`, `marital`, `education`, etc.


In [21]:
logistic_configs = [
    {"name": "config_1",  "penalty": "l2",         "C": 1.0,  "solver": "lbfgs",     "max_iter": 1000, "random_state": 42},
    {"name": "config_2",  "penalty": "l2",         "C": 0.1,  "solver": "lbfgs",     "max_iter": 1000, "random_state": 42},
    {"name": "config_3",  "penalty": "l2",         "C": 10.0, "solver": "lbfgs",     "max_iter": 1000, "random_state": 42},
    {"name": "config_4",  "penalty": "l1",         "C": 1.0,  "solver": "liblinear", "max_iter": 1000, "random_state": 42},
    {"name": "config_5",  "penalty": "l1",         "C": 0.5,  "solver": "liblinear", "max_iter": 1500, "random_state": 42},
    {"name": "config_6",  "penalty": "l2",         "C": 5.0,  "solver": "saga",      "max_iter": 2000, "random_state": 42},
    {"name": "config_7",  "penalty": "l2",         "C": 0.01, "solver": "saga",      "max_iter": 2000, "random_state": 42},
    {"name": "config_8",  "penalty": "elasticnet", "C": 1.0,  "solver": "saga",      "max_iter": 3000, "random_state": 42},
    {"name": "config_9",  "penalty": None,         "C": 1.0,  "solver": "lbfgs",     "max_iter": 1000, "random_state": 42},
    {"name": "config_10", "penalty": "l2",         "C": 5.0,  "solver": "lbfgs",     "max_iter": 2000, "random_state": 42}
]

## 8. Entrenamiento, evaluación y registro en MLflow

El modelo se registrará con el nombre:

```text
linear_regretion_pipeline
```

Luego se podrá cargar desde Streamlit con:

```python
mlflow.sklearn.load_model("models:/linearRegretion02/1")
```


In [22]:
for config in logistic_configs:

    model = LogisticRegression(
        **{k: v for k, v in config.items() if k != "name"}
    )
    params_max_iter = 1000

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
            ("num", "passthrough", columnas_numericas)
        ]
    )
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("modelo", LogisticRegression(
                 penalty=config["penalty"],
                 C=config["C"],
                 solver=config["solver"],
                 max_iter=config["max_iter"],
                 random_state=config["random_state"]    
            ))
        ]
    )

    with mlflow.start_run(run_name="linear_regretion_pipeline_base") as run:
        
        pipeline.fit(X_train, y_train)
    
        y_pred_train = pipeline.predict(X_train)
        y_pred_test = pipeline.predict(X_test)
    
        acc_training = accuracy_score(y_train, y_pred_train)
        acc_test = accuracy_score(y_test, y_pred_test)
        
     
        precision_training = precision_score(y_train, y_pred_train)
        precision_test = precision_score(y_test, y_pred_test)
        
        
        recall_training = recall_score(y_train, y_pred_train)
        recall_test = recall_score(y_test, y_pred_test)
        
    
        f1_training = f1_score(y_train, y_pred_train)
        f1_test = f1_score(y_test, y_pred_test)
    
        mlflow.log_param("modelo", "LogisticRegression")
        mlflow.log_param("penalty", config["penalty"])
        mlflow.log_param("C", config["C"])
        mlflow.log_param("solver", config["solver"])
        mlflow.log_param("max_iter", config["max_iter"])
        mlflow.log_param("random_state", config["random_state"])
        # Parámetros del experimento
        mlflow.log_param("dataset", "estudiantes.csv")
        mlflow.log_param("target", "y")
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("removed_column", "beca")
        mlflow.log_param("categorical_encoder", "OneHotEncoder")
        mlflow.log_param("handle_unknown", "ignore")
    
        mlflow.log_metric("accuracy_training", acc_training)
        mlflow.log_metric("accuracy_test", acc_test)
    
        mlflow.log_metric("precision_training", precision_training)
        mlflow.log_metric("precision_test", precision_test)
    
        mlflow.log_metric("recall_training", recall_training)
        mlflow.log_metric("recall_test", recall_test)
    
        mlflow.log_metric("f1_training", f1_training)
        mlflow.log_metric("f1_test", f1_test)
    
    
        input_example = X_train.head(5)
    
        signature = infer_signature(
            input_example,
            pipeline.predict(input_example)
        )
    
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="modelo_student_pipeline",
            registered_model_name="logisticRegression01",
            input_example=input_example,
            signature=signature
        )
    
        print("RUN ID:", run.info.run_id)
    
        print(f"Accuracy Train: {acc_training:.4f}")
        print(f"Accuracy Test : {acc_test:.4f}")
    


/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (trainin

RUN ID: 386498c4459a41ce968849c5b19931ac
Accuracy Train: 0.8173
Accuracy Test : 0.8210
🏃 View run linear_regretion_pipeline_base at: http://127.0.0.1:9090/#/experiments/1/runs/386498c4459a41ce968849c5b19931ac
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/1


/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/16 08:42:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 08:42:27 WARNING mlflow.sklearn: Saving s

RUN ID: 157b4d3cdc714ba1b1f578e14eeb2522
Accuracy Train: 0.8173
Accuracy Test : 0.8210
🏃 View run linear_regretion_pipeline_base at: http://127.0.0.1:9090/#/experiments/1/runs/157b4d3cdc714ba1b1f578e14eeb2522
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/1


2026/05/16 08:42:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'logisticRegression01' already exists. Creating a new version of this model...
2026/05/16 08:42:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: logisticRegression01, version 4
Created version '4' of model 'logisticRegression01'.
/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warnin

...

2026/05/16 08:42:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'logisticRegression01' already exists. Creating a new version of this model...
2026/05/16 08:42:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: logisticRegression01, version 10
Created version '10' of model 'logisticRegression01'.
/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warn

RUN ID: 99a03dbefcbd40d995495fb322ce698f
Accuracy Train: 0.8173
Accuracy Test : 0.8210
🏃 View run linear_regretion_pipeline_base at: http://127.0.0.1:9090/#/experiments/1/runs/99a03dbefcbd40d995495fb322ce698f
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/1


/Users/marcovinicioguapiguapi/Desktop/MaestriaAI/ClasesB1/HerramientasAi/LabTasks/TaskMlFlow/envlab/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/16 08:42:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 08:42:54 WARNING mlflow.sklearn: Saving s

RUN ID: 69dc1a4ee6624b54afc6a1b680d0739f
Accuracy Train: 0.8173
Accuracy Test : 0.8210
🏃 View run linear_regretion_pipeline_base at: http://127.0.0.1:9090/#/experiments/1/runs/69dc1a4ee6624b54afc6a1b680d0739f
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/1


Created version '11' of model 'logisticRegression01'.
